# Tableau détaillé par point de données — `blob_nn_4x10-1` (gain vs baseline)

Ce notebook fusionne tous les `results.csv` (un par combo de bound strategy) et produit
**une ligne par point de données** (`data_index`, `target`) pour chaque combo — plus
d'agrégation/moyenne, chaque run individuel est conservé.

Colonnes produites :
`combo, strategy, l, u_idx, k, j_idx, data_index, target, certified, optimal_value, gain, LB_neuron1, UB_neuron1, LB_neuron2, UB_neuron2`

**Règles retenues** :
- `certified` = `optimal_value > 0` (par ligne).
- `results.csv` et `stable_actives_study.csv` sont tronqués à **78 lignes max** ; un
  combo avec **moins de 78 lignes** est **écarté**.
- **`gain`** = `optimal_value_combo − optimal_value_baseline` pour **ce point de
  données précis**, en appariant chaque ligne du combo à la ligne du **baseline
  `combo_0000`** ayant le **même `data_index` et le même `target`**.
  → **Hypothèse à confirmer** : `(data_index, target)` identifie une ligne de façon
  unique dans `results.csv`. Si ce n'est pas le cas, dites-le-moi et j'ajoute
  `epsilon` à la clé de jointure.
- **`LB_neuron1`/`UB_neuron1`/`LB_neuron2`/`UB_neuron2`** proviennent de
  `stable_actives_study.csv`, appariées par **`data_index`** uniquement (les bornes
  ne dépendent que de l'entrée, pas du `target`).
- Le baseline `combo_0000` apparaît lui aussi ligne par ligne, avec `gain = 0`.


In [9]:
import re
from pathlib import Path

import pandas as pd

# === CONFIGURATION ===

# Racine contenant tous les dossiers de runs (un par combo).
# Adaptez ce chemin à votre arborescence WSL, ex:
# BENCHMARK_ROOT = Path("/mnt/c/Users/vous/.../results/benchmark/blob_nn_4x10-1")
BENCHMARK_ROOT = Path("../results/benchmark/blob_nn_4x10-1")

# Nom du dossier de run, ex:
#   2026_07_08_15h28_19s_test__combo_0000__baseline__all_one_variable
#   2026_07_09_01h54_43s_test__combo_0001__product_0__l0_u0_k1_j0__composed
#   2026_07_14_14h56_49s_your_run_title__combo_0232__product_231__l2_u9_k3_j0__composed
RUN_DIR_RE = re.compile(r"combo_(\d+)__(.+)$")
PRODUCT_IDX_RE = re.compile(r"l(\d+)_u(\d+)_k(\d+)_j(\d+)")

# Un combo est gardé seulement s'il a au moins ce nombre de lignes utilisables ;
# au-delà, on tronque aux MAX_RUNS premières lignes ; en dessous, le combo est écarté.
MIN_RUNS = 78
MAX_RUNS = 78

# Clé utilisée pour apparier chaque ligne d'un combo à la ligne correspondante du
# baseline (combo_0000), afin de calculer le gain.
RESULTS_MERGE_KEYS = ["data_index", "target"]

# Clé utilisée pour apparier les bornes LB/UB de stable_actives_study.csv
# (les bornes ne dépendent que de l'entrée, pas du target).
STABLE_MERGE_KEY = "data_index"


In [10]:
def parse_run_dir(dirname: str) -> dict:
    """Extrait combo_id, stratégie, et les indices (l,u,k,j) si présents."""
    m = RUN_DIR_RE.search(dirname)
    if not m:
        return None

    combo_id = int(m.group(1))
    rest = m.group(2)  # ex: "product_0__l0_u0_k1_j0__composed" ou "baseline__all_one_variable"

    idx_match = PRODUCT_IDX_RE.search(rest)
    if idx_match:
        l, u, k, j = map(int, idx_match.groups())
        strategy = rest.rsplit("__", 1)[-1]  # dernier segment = nom de la stratégie
    else:
        l = u = k = j = None
        strategy = "none" if combo_id == 0 else rest

    return {
        "combo": f"combo_{combo_id:04d}",
        "combo_id": combo_id,
        "strategy": strategy,
        "l": l, "u_idx": u, "k": k, "j_idx": j,
    }


def find_run_dirs(root: Path) -> list[dict]:
    """Retourne la liste des runs trouvés avec leurs métadonnées et chemins de fichiers."""
    if not root.exists():
        raise FileNotFoundError(f"Dossier introuvable: {root.resolve()}")

    runs = []
    for d in sorted(root.glob("*__combo_*")):
        if not d.is_dir():
            continue
        meta = parse_run_dir(d.name)
        if meta is None:
            print(f"[!] Nom de dossier non reconnu, ignoré: {d.name}")
            continue

        results_csv = d / "results.csv"
        stable_csv = d / "stable_actives_study.csv"

        if not results_csv.exists():
            print(f"[!] Pas de results.csv dans {d.name}")
            continue
        if not stable_csv.exists():
            print(f"[!] Pas de stable_actives_study.csv dans {d.name}")

        meta["results_csv"] = results_csv
        meta["stable_csv"] = stable_csv if stable_csv.exists() else None
        meta["dirname"] = d.name
        runs.append(meta)

    return sorted(runs, key=lambda r: r["combo_id"])


runs = find_run_dirs(BENCHMARK_ROOT)
print(f"{len(runs)} run(s) trouvé(s):")
for r in runs:
    print(f"  {r['combo']} ({r['strategy']}) l={r['l']} u={r['u_idx']} k={r['k']} j={r['j_idx']}")


[!] Pas de results.csv dans 2026_07_10_13h31_53s_test__combo_0000__baseline__all_one_variable
[!] Pas de results.csv dans your_run_title__combo_0163__product_162__l2_u1_k4_j0__composed
[!] Pas de results.csv dans your_run_title__combo_0247__product_246__l3_u1_k4_j0__composed
178 run(s) trouvé(s):
  combo_0000 (none) l=None u=None k=None j=None
  combo_0000 (none) l=None u=None k=None j=None
  combo_0000 (none) l=None u=None k=None j=None
  combo_0000 (none) l=None u=None k=None j=None
  combo_0001 (composed) l=0 u=0 k=1 j=0
  combo_0001 (composed) l=0 u=0 k=1 j=0
  combo_0002 (composed) l=0 u=0 k=1 j=1
  combo_0002 (composed) l=0 u=0 k=1 j=1
  combo_0002 (composed) l=0 u=0 k=1 j=1
  combo_0003 (composed) l=0 u=0 k=1 j=2
  combo_0003 (composed) l=0 u=0 k=1 j=2
  combo_0004 (composed) l=0 u=0 k=2 j=0
  combo_0004 (composed) l=0 u=0 k=2 j=0
  combo_0005 (composed) l=0 u=0 k=2 j=1
  combo_0006 (composed) l=0 u=0 k=2 j=2
  combo_0006 (composed) l=0 u=0 k=2 j=2
  combo_0007 (composed) l=0 u=

In [11]:
def load_limited_csv(csv_path: Path, min_rows: int = MIN_RUNS, max_rows: int = MAX_RUNS) -> pd.DataFrame | None:
    """Charge un CSV, le tronque à max_rows lignes, ou renvoie None s'il en a moins que min_rows."""
    df = pd.read_csv(csv_path)
    n_original = len(df)
    if n_original < min_rows:
        return None
    if n_original > max_rows:
        df = df.iloc[:max_rows].reset_index(drop=True)
    assert len(df) <= max_rows, (
        f"BUG: {csv_path} a {len(df)} lignes après troncature (attendu <= {max_rows})"
    )
    return df


In [12]:
# === Baseline (combo_0000) ===
baseline_run = next((r for r in runs if r["combo_id"] == 0), None)
if baseline_run is None:
    raise RuntimeError("Aucun combo_0000 (baseline) trouvé — impossible de calculer le gain.")

baseline_results_df = load_limited_csv(baseline_run["results_csv"])
if baseline_results_df is None:
    raise RuntimeError(
        f"combo_0000 a moins de {MIN_RUNS} lignes utilisables — impossible de calculer un gain fiable."
    )

missing_keys = [k for k in RESULTS_MERGE_KEYS if k not in baseline_results_df.columns]
if missing_keys:
    raise RuntimeError(f"Colonnes de jointure manquantes dans results.csv du baseline: {missing_keys}")

print(f"Baseline: {baseline_run['combo']} ({baseline_run['strategy']}) — {len(baseline_results_df)} lignes")


RuntimeError: combo_0000 a moins de 78 lignes utilisables — impossible de calculer un gain fiable.

In [ ]:
def per_point_table(r: dict, baseline_results_df: pd.DataFrame) -> pd.DataFrame | None:
    """Construit une ligne par point de données pour un combo donné, avec gain et bornes LB/UB."""
    results_df = load_limited_csv(r["results_csv"])
    if results_df is None:
        return None

    keep_cols = RESULTS_MERGE_KEYS + ["optimal_value"]
    df = results_df[keep_cols].copy()
    df["certified"] = results_df["optimal_value"] > 0

    # Gain point par point vs baseline (jointure sur data_index + target)
    merged = df.merge(
        baseline_results_df[RESULTS_MERGE_KEYS + ["optimal_value"]],
        on=RESULTS_MERGE_KEYS,
        suffixes=("", "_baseline"),
        how="left",
    )
    if merged["optimal_value_baseline"].isna().any():
        n_missing = merged["optimal_value_baseline"].isna().sum()
        print(f"[!] {r['combo']}: {n_missing} ligne(s) sans correspondance dans le baseline sur {RESULTS_MERGE_KEYS}")
    merged["gain"] = merged["optimal_value"] - merged["optimal_value_baseline"]
    merged = merged.drop(columns=["optimal_value_baseline"])

    # Bornes LB/UB par data_index, depuis stable_actives_study.csv
    stable_df = load_limited_csv(r["stable_csv"]) if r["stable_csv"] is not None else None
    lb1_col = f"LB_Layer_{r['l']}_Neuron_{r['u_idx']}" if r["l"] is not None else None
    ub1_col = f"UB_Layer_{r['l']}_Neuron_{r['u_idx']}" if r["l"] is not None else None
    lb2_col = f"LB_Layer_{r['k']}_Neuron_{r['j_idx']}" if r["k"] is not None else None
    ub2_col = f"UB_Layer_{r['k']}_Neuron_{r['j_idx']}" if r["k"] is not None else None

    for col_name, source_col in [
        ("LB_neuron1", lb1_col), ("UB_neuron1", ub1_col),
        ("LB_neuron2", lb2_col), ("UB_neuron2", ub2_col),
    ]:
        if stable_df is None or source_col is None or source_col not in stable_df.columns:
            merged[col_name] = float("nan")
        else:
            bounds = stable_df[[STABLE_MERGE_KEY, source_col]].rename(columns={source_col: col_name})
            merged = merged.merge(bounds, on=STABLE_MERGE_KEY, how="left")

    merged.insert(0, "combo", r["combo"])
    merged.insert(1, "strategy", r["strategy"])
    merged.insert(2, "l", r["l"])
    merged.insert(3, "u_idx", r["u_idx"])
    merged.insert(4, "k", r["k"])
    merged.insert(5, "j_idx", r["j_idx"])

    return merged


In [ ]:
tables = []
skipped = []

for r in runs:
    t = per_point_table(r, baseline_results_df)
    if t is None:
        skipped.append(r["combo"])
        continue
    tables.append(t)

if skipped:
    print(f"{len(skipped)} combo(s) écarté(s) (moins de {MIN_RUNS} runs): {skipped}")

detail_df = pd.concat(tables, ignore_index=True)

# Vérification explicite : aucun combo ne doit dépasser MAX_RUNS lignes.
counts = detail_df.groupby("combo").size()
bad = counts[counts > MAX_RUNS]
if not bad.empty:
    raise AssertionError(f"Combos dépassant MAX_RUNS lignes après troncature: {bad.to_dict()}")

detail_df = detail_df[
    [
        "combo", "strategy", "l", "u_idx", "k", "j_idx",
        "data_index", "target", "certified", "optimal_value", "gain",
        "LB_neuron1", "UB_neuron1", "LB_neuron2", "UB_neuron2",
    ]
]
detail_df


## Export (CSV, une ligne par point de données)

In [ ]:
OUTPUT_CSV = "combo_detail_by_datapoint.csv"
detail_df.to_csv(OUTPUT_CSV, index=False)
print(f"Tableau exporté vers {OUTPUT_CSV} ({len(detail_df)} lignes)")
